In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import netCDF4
from dask import array as da
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import BoundaryNorm

import geopandas as gp
from geopy import distance

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.util import add_cyclic_point

import glob
from tqdm import tqdm

import re
from scipy.stats import gmean
import cmocean

In [2]:
def calc_distance(x):
    return distance.distance(
        x[['OBS_CENTROID_LAT', 'OBS_CENTROID_LON']].values,
        x[['FCST_CENTROID_LAT', 'FCST_CENTROID_LON']].values).km

def analyze_pkl(df):
    cluster_pair_str = 'CF0*[1-9][0-9]*_CO0*[1-9][0-9]*'
    cluster_pairs = df[df['OBJECT_ID'].str.fullmatch(cluster_pair_str)].copy(deep=True)
    cluster_pairs['CLUSTER_IDX'] = cluster_pairs['OBJECT_ID'].str[-3:].astype(int)
    cluster_pairs['INTERSECTION_OVER_UNION'] = cluster_pairs['INTERSECTION_AREA'] / cluster_pairs['UNION_AREA']
    var_cols = [
        'CENTROID_DIST',
        'INTERSECTION_OVER_UNION',
    ]
    cols_shared = [
        'CLUSTER_IDX',
        'INIT',
        'MEMBER',
        'LEAD',
        'INIT_MONTH',
        'VALID_MONTH'
    ]
    df_pairs = cluster_pairs[np.concatenate([cols_shared, var_cols])]

    fcst_cluster_str = 'CF0*[1-9][0-9]*'
    obs_cluster_str = 'CO0*[1-9][0-9]*'
    fcst_clus_matched = df[df['OBJECT_ID'].str.fullmatch(fcst_cluster_str)].copy(deep=True)
    obs_clus_matched = df[df['OBJECT_ID'].str.fullmatch(obs_cluster_str)].copy(deep=True)
    fcst_clus_matched['CLUSTER_IDX'] = fcst_clus_matched['OBJECT_ID'].str.lstrip('CF0*').astype(int)
    obs_clus_matched['CLUSTER_IDX'] = obs_clus_matched['OBJECT_ID'].str.lstrip('CO0*').astype(int)
    cols = [
        'AREA',
        'INTENSITY_50',
        'CENTROID_LAT',
        'CENTROID_LON'
    ]
    fcst_cols = fcst_clus_matched[np.concatenate([cols_shared, cols])].rename(columns={col: 'FCST_'+col for col in cols})
    obs_cols = obs_clus_matched[np.concatenate([cols_shared, cols])].rename(columns={col: 'OBS_'+col for col in cols})
    df_clus = pd.merge(fcst_cols, obs_cols)
    for col in ['AREA', 'INTENSITY_50']:
        df_clus[f'LOG_{col}_RATIO'] = np.log(df_clus[f'FCST_{col}'] / df_clus[f'OBS_{col}'])
    df_clus['CENTROID_DIST_KM'] = df_clus.apply(lambda x: calc_distance(x), axis=1)
    df_clus['ABS_LOG_AREA_RATIO'] = abs(df_clus['LOG_AREA_RATIO'])
    df_clus['ABS_LOG_INTENSITY_50_RATIO'] = abs(df_clus['LOG_INTENSITY_50_RATIO'])

    return pd.merge(df_pairs.reset_index().drop(columns=['index']), df_clus)

#### Load pkl files

In [4]:
MODE_output = '/glade/work/jtcohen/MODE_files_final/saved_output'
df_random = pd.read_pickle(f'{MODE_output}/mode_1989-2018_SMYLE_OISST_24L_ocetrac_random.pkl')
df = pd.read_pickle(f'{MODE_output}/mode_1989-2018_SMYLE_OISST_24L_ocetrac_r3.pkl')
df_FOSI = pd.read_pickle('/glade/work/jtcohen/MODE_files_final/output_r3/mode_1989-2018_SMYLE_FOSI_24L_ocetrac.pkl')

In [5]:
%%time
df_random_final = analyze_pkl(df_random)
df_final = analyze_pkl(df)

CPU times: user 3min 41s, sys: 3.1 s, total: 3min 45s
Wall time: 3min 57s


In [6]:
vars = ['CENTROID_DIST_KM', 'INTERSECTION_OVER_UNION', 'FCST_AREA', 'OBS_AREA']
log_vars = ['ABS_LOG_AREA_RATIO', 'ABS_LOG_INTENSITY_50_RATIO']

In [7]:
ds_attrs = df_final.set_index(
        ['INIT', 'MEMBER', 'LEAD', 'CLUSTER_IDX']
    )[vars+log_vars].to_xarray().rename(
    {'INIT': 'init',
     'MEMBER': 'member',
     'LEAD': 'lead',
     'CLUSTER_IDX': 'clus_id'}
)

ds_attrs_random = df_random_final.set_index(
        ['INIT', 'MEMBER', 'LEAD', 'CLUSTER_IDX']
    )[vars+log_vars].to_xarray().rename(
    {'INIT': 'init',
     'MEMBER': 'member',
     'LEAD': 'lead',
     'CLUSTER_IDX': 'clus_id'}
)

#### Load nc files

In [8]:
files = [f'{MODE_output}/object_footprints_r3_1989-2003.nc', f'{MODE_output}/object_footprints_r3_2004-2018.nc']
files_random = [f'{MODE_output}/object_footprints_random_1989-2003.nc', f'{MODE_output}/object_footprints_random_2004-2018.nc']
chunks = {
    'lat': -1,
    'lon': -1,
    'lead': 1,
    'member': 10,
    'init': 20,
}
ds = xr.open_mfdataset(files)
ds_random = xr.open_mfdataset(files_random)

In [9]:
ds = ds['fcst_clus_id'].chunk(chunks)
ds_random = ds_random['fcst_clus_id'].chunk(chunks)

## Map attributes

In [10]:
def assign_values(map, values):
    """
    Assign cluster attribute values to points with that cluster id in the map.
    """
    return xr.where(map==values['clus_id'], values, np.nan).mean('clus_id', skipna=True)

In [12]:
attr_names = {
    'CENTROID_DIST_KM': 'Centroid Distance',
    'INTERSECTION_OVER_UNION': 'Intersection Over Union',
    'ABS_LOG_AREA_RATIO': 'Log Area Ratio',
    'ABS_LOG_INTENSITY_50_RATIO': 'Log Median Intensity Ratio'
}

In [13]:
attr_strs = vars + log_vars
lead = 3
footprint_lead = ds.isel(lead=lead)
footprint_lead_random = ds_random.isel(lead=lead)
attrs_lead = ds_attrs.isel(lead=lead)
attrs_lead_random = ds_attrs_random.isel(lead=lead)

In [35]:
%%time
mapped_attrs_random = assign_values(footprint_lead_random, attrs_lead_random).groupby('init.month').mean().mean('member').load()

CPU times: user 1min 7s, sys: 43.5 s, total: 1min 51s
Wall time: 2min 47s


In [30]:
%%time
mapped_attrs_lead3 = assign_values(footprint_lead, attrs_lead).groupby('init.month').mean().mean('member').load()

CPU times: user 1min 27s, sys: 51.5 s, total: 2min 18s
Wall time: 3min 1s


In [34]:
mapped_attrs = [assign_values(ds.isel(lead=lead), ds_attrs.isel(lead=lead)).groupby('init.month').mean().mean('member').load() for lead in tqdm(range(24))]

100%|██████████| 24/24 [1:02:16<00:00, 155.69s/it]


In [36]:
ds_mapped_attrs = xr.concat(mapped_attrs, dim='lead')
ds_mapped_attrs['lead'] = ds['lead']

In [37]:
ds_mapped_attrs.to_netcdf('/glade/u/home/jtcohen/MODE/notebooks/GRL_code/final_data/spatial_attrs_r3.nc')

In [38]:
mapped_attrs_random.to_netcdf('/glade/u/home/jtcohen/MODE/notebooks/GRL_code/final_data/spatial_attrs_random_03L.nc')